# Experiment 3 - Module 3: Structural Relaxation & Induced-Fit Docking
### Two-Phase Pipeline: GPU Relaxation (Steps 1–9) + CPU Docking (Steps 10–13)

This notebook implements **Module 3** of the Experiment 3 engineering roadmap, split cleanly so GPU-intensive steps run on GPU and CPU-bound docking runs on any CPU instance:

- **Phase 1 — GPU Instance (Steps 1–9 + 9b Checkpoint Export):**
  1. **ESMFold Tertiary Structure Prediction & pLDDT Gating:** High-throughput 3D folding (`mean pLDDT >= 80.0`).
  2. **OpenMM / AMBER14SB Constrained Energy Relaxation (GPU/CUDA):** Heavy backbone atoms ($N, C\alpha, C, O$) restrained ($k = 10\text{ kcal/mol}\cdot\text{\AA}^2$) so side chains resolve clashes in ~8s/design on GPU.
  3. **Foldseek Structural Alignment:** Superposition against **IsPETase (6EQE)** and **LCC (4EB0)** (`--alignment-type 1`).
  4. **3D Active-Site Architecture Audit & Ligand Prep:** Maps triad, oxyanion hole, Trp clamp, `--flexres`, and prepares 2-HE(MHET) & 3PET ligands.
  5. **Checkpoint Export (Step 9b):** Saves `exp3_module_3_gpu_checkpoint.zip` (and syncs to Google Drive) for handoff.

- **Phase 2 — CPU Instance (Step 10a Restore + Steps 10–13):**
  6. **CPU Setup & Checkpoint Restore (Step 10a):** Installs only lightweight docking tools (`smina`, `openbabel`, `rdkit`) and restores the GPU checkpoint.
  7. **Flexible Side-Chain Smina Docking (`--flexres`):** Evaluates all 5 modes within $\le 1.5\text{ kcal/mol}$ for strict Near-Attack Conformation (NAC) and Bürgi-Dunitz trajectory ($\theta \in [95^\circ, 115^\circ]$, $d_{\text{attack}} \le 3.5\text{ \AA}$, $\Delta G_{\text{bind}} \le -5.0\text{ kcal/mol}$).
  8. **Disulfide Topology, Visualization & Export:** Audits disulfides, plots summary charts, and archives final results.


## 1. Environment & GPU Verification
Check hardware accelerator (NVIDIA L4 / A100 / T4 recommended) and mount Google Drive.


In [1]:
import subprocess, os, sys, pathlib

# Check GPU specifications
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                              capture_output=True, text=True).stdout
    print("GPU Detected:\n" + gpu_info)
except Exception as e:
    print("Warning: nvidia-smi failed or no GPU found:", e)

# Mount Google Drive if running in Google Colab
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully at /content/drive.")

# Setup base workspace directories
WORKSPACE = pathlib.Path("exp3_module_3_workspace")
WORKSPACE.mkdir(exist_ok=True)
PDB_DIR = WORKSPACE / "esmfold_pdbs"
RELAX_DIR = WORKSPACE / "relaxed_pdbs"
DOCK_DIR = WORKSPACE / "docking_results"
LIGAND_DIR = WORKSPACE / "docking_ligands"
REFS_DIR = WORKSPACE / "references"

for d in [PDB_DIR, RELAX_DIR, DOCK_DIR, LIGAND_DIR, REFS_DIR]:
    d.mkdir(exist_ok=True)

print("Workspace initialized in:", WORKSPACE.resolve())


GPU Detected:
name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07

Mounted at /content/drive
Google Drive mounted successfully at /content/drive.
Workspace initialized in: /content/exp3_module_3_workspace


## 2. Install Dependencies & Structural Tools (OpenMM, Foldseek, Smina, Meeko, RDKit)


In [2]:
!apt-get update -qq
!apt-get install -y -qq libboost-all-dev swig openbabel curl wget

# Install core scientific and biophysical packages (matching Experiment 2 setup with meeko & openmm)
!pip install -q vina
!pip install -q transformers accelerate torch biopython matplotlib pandas numpy rdkit "meeko<=0.4.0" scipy openmm pdbfixer

import urllib.request, tarfile, shutil

# 1. Download and unpack static Foldseek binary (Linux AVX2)
FOLDSEEK_URL = "https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz"
foldseek_bin = pathlib.Path("foldseek/bin/foldseek")
if not foldseek_bin.exists():
    print("Downloading Foldseek static Linux binary...")
    urllib.request.urlretrieve(FOLDSEEK_URL, "foldseek.tar.gz")
    with tarfile.open("foldseek.tar.gz", "r:gz") as tar:
        tar.extractall(".")
    print("Foldseek installed.")

!chmod +x foldseek/bin/foldseek
!foldseek/bin/foldseek version

# 2. Download and verify static Smina binary for molecular docking
smina_dest = pathlib.Path("/usr/local/bin/smina")
if not smina_dest.exists():
    print("Downloading static Smina binary...")
    !curl -fsSL "https://sourceforge.net/projects/smina/files/smina.static/download" -o /usr/local/bin/smina
    !chmod +x /usr/local/bin/smina

!/usr/local/bin/smina --version
print("Structural tools verified successfully.")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package javascript-common.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../000-javascript-common_11+nmu1_all.deb ...
Unpacking javascript-common (11+nmu1) ...
Selecting previously unselected package libboost1.83-tools-dev.
Preparing to unpack .../001-libboost1.83-tools-dev_1.83.0-2.1ubuntu3.2_amd64.deb ...
Unpacking libboost1.83-tools-dev (1.83.0-2.1ubuntu3.2) ...
Selecting previously unselected package libboost-tools-dev.
Preparing to unpack .../002-libboost-tools-dev_1.83.0.1ubuntu2_amd64.deb ...
Unpacking libboost-tools-dev (1.83.0.1ubuntu2) ...
Selecting previously unselected package libboost-atomic1.83.0:amd64.
Preparing to unpack .../003-libboost-atomic1.83.0_1.83.0-2.1ubunt

/tmp/ipykernel_461/1479579029.py:17: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(".")


Foldseek installed.
4f5c00ccb5590f08a261b9144fd9a5d28a71fa0e
Smina Oct 15 2019.  Based on AutoDock Vina 1.1.2.
Structural tools verified successfully.


## 3. Fetch Crystallographic Reference Standards: IsPETase (6EQE) and LCC (4EB0)


In [3]:
import urllib.request

refs = {
    "6EQE": "https://files.rcsb.org/download/6EQE.pdb",
    "4EB0": "https://files.rcsb.org/download/4EB0.pdb"
}

for pdb_id, url in refs.items():
    dest = REFS_DIR / f"{pdb_id}.pdb"
    if not dest.exists():
        print(f"Fetching {pdb_id} from RCSB PDB...")
        urllib.request.urlretrieve(url, dest)
        print(f"Saved: {dest}")

print("Reference PDBs ready in:", list(REFS_DIR.glob("*.pdb")))


Fetching 6EQE from RCSB PDB...
Saved: exp3_module_3_workspace/references/6EQE.pdb
Fetching 4EB0 from RCSB PDB...
Saved: exp3_module_3_workspace/references/4EB0.pdb
Reference PDBs ready in: [PosixPath('exp3_module_3_workspace/references/6EQE.pdb'), PosixPath('exp3_module_3_workspace/references/4EB0.pdb')]


## 4. Load Candidate Sequences & Tier 1 Sequence Quality Gate
Loads candidates from Module 2.
Verifies that every candidate possesses:
1. The canonical nucleophile elbow pentapeptide motif (`[GWA]-x-S-x-G` or `G-x-S-x-G`).
2. Essential catalytic triad residues: at least one Aspartate (`D`) and Histidine (`H`).


In [4]:
import re, pandas as pd

# Path resolution: checks local inputs, Drive, or uploads
candidate_fasta_path = pathlib.Path("inputs/designs_exp3_screened_novel.faa")

if not candidate_fasta_path.exists():
    possible_paths = list(pathlib.Path(".").glob("**/designs_exp3_screened_novel.faa")) + \
                     list(pathlib.Path("/content/drive").glob("**/designs_exp3_screened_novel.faa"))
    if possible_paths:
        candidate_fasta_path = possible_paths[0]
        print(f"Found candidate FASTA at: {candidate_fasta_path}")

if not candidate_fasta_path.exists():
    raise FileNotFoundError("Could not locate designs_exp3_screened_novel.faa. Please upload it or place it in inputs/.")

def parse_fasta(path):
    recs = []
    curr_id, curr_desc, curr_seq = None, "", []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if curr_id:
                    recs.append({"id": curr_id, "desc": curr_desc, "sequence": "".join(curr_seq), "length": len("".join(curr_seq))})
                curr_id = line[1:].split()[0]
                curr_desc = line[1:]
                curr_seq = []
            else:
                curr_seq.append(line.upper().replace("*", ""))
        if curr_id:
            recs.append({"id": curr_id, "desc": curr_desc, "sequence": "".join(curr_seq), "length": len("".join(curr_seq))})
    return recs

raw_records = parse_fasta(candidate_fasta_path)
print(f"Loaded {len(raw_records)} novel de novo designs from Module 2.")

tier1_passed = []
for r in raw_records:
    seq = r["sequence"]
    motif = re.search(r'G[HWYFQ]S.G', seq) or re.search(r'[GWA].[SMG].G', seq) or re.search(r'G.S.G', seq)
    has_d = 'D' in seq
    has_h = 'H' in seq
    passed = (motif is not None) and has_d and has_h
    r["motif_found"] = bool(motif)
    r["motif_match"] = motif.group(0) if motif else "None"
    r["has_triad_residues"] = has_d and has_h
    r["passed_tier1"] = passed
    if passed:
        tier1_passed.append(r)

df_tier1 = pd.DataFrame(raw_records)
print(f"\nTier 1 Sequence Gate Results:")
print(f"  Total Candidates Evaluated: {len(df_tier1)}")
print(f"  Passed (Motif + Asp + His present): {len(tier1_passed)} / {len(df_tier1)} ({100*len(tier1_passed)/max(len(df_tier1),1):.1f}%)")


Loaded 140 novel de novo designs from Module 2.

Tier 1 Sequence Gate Results:
  Total Candidates Evaluated: 140
  Passed (Motif + Asp + His present): 140 / 140 (100.0%)


## 5. Tier 2: ESMFold Tertiary Structure Prediction & pLDDT Evaluation
Predicts 3D structures using `facebook/esmfold_v1` on GPU (or ESM Atlas API as fallback).
Caches coordinates to `esmfold_pdbs/{id}.pdb` to prevent recomputing folded structures upon re-runs.
Automatically detects pre-existing or uploaded PDB directories/archives and skips folding when structures are already present.
Filters designs with $\text{mean pLDDT} \ge 80.0$.


In [5]:
import torch, time, urllib.request, zipfile, shutil
from transformers import AutoTokenizer, EsmForProteinFolding

# 1. Detect and extract uploaded PDB coordinate archives or local directories
uploaded_zips = list(pathlib.Path(".").glob("*.zip")) +                 list(pathlib.Path("inputs").glob("*.zip")) +                 list(pathlib.Path("/content").glob("*.zip") if pathlib.Path("/content").exists() else [])
for zpath in uploaded_zips:
    if "pdb" in zpath.name.lower():
        print(f"Found uploaded coordinate archive: {zpath}. Extracting to {PDB_DIR}...")
        try:
            with zipfile.ZipFile(zpath, 'r') as zip_ref:
                for member in zip_ref.namelist():
                    if member.endswith(".pdb"):
                        fname = pathlib.Path(member).name
                        target_file = PDB_DIR / fname
                        if not target_file.exists():
                            with zip_ref.open(member) as source, open(target_file, "wb") as target:
                                shutil.copyfileobj(source, target)
        except Exception as e:
            print(f"Archive extraction notice: {e}")

# Check alternate directories where PDBs may have been uploaded
alt_dirs = [
    pathlib.Path("esmfold_pdbs"),
    pathlib.Path("inputs/esmfold_pdbs"),
    pathlib.Path("pdbs"),
    pathlib.Path("/content/esmfold_pdbs"),
    pathlib.Path("/content/pdbs")
]
for adir in alt_dirs:
    if adir.exists() and adir.resolve() != PDB_DIR.resolve():
        for pfile in adir.glob("*.pdb"):
            dest_file = PDB_DIR / pfile.name
            if not dest_file.exists():
                shutil.copy(pfile, dest_file)

# 2. Check existing PDB models against Tier 1 passing candidates
needed_ids = [r["id"] for r in tier1_passed]
existing_ids = {p.stem for p in PDB_DIR.glob("*.pdb") if p.stat().st_size > 0}
missing_ids = [qid for qid in needed_ids if qid not in existing_ids]

print(f"Candidate PDB status in {PDB_DIR}: {len(needed_ids) - len(missing_ids)} / {len(needed_ids)} models ready.")

use_hf = False
tokenizer = None
model = None
device = "cuda" if torch.cuda.is_available() else "cpu"

if len(missing_ids) == 0:
    print(f"All {len(needed_ids)} candidate PDB structures already exist in {PDB_DIR}!")
    print("SKIPPING ESMFold model loading and folding entirely to save time and GPU memory.")
else:
    print(f"Missing {len(missing_ids)} structures. Preparing ESMFold v1 model on {device}...")
    try:
        tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
        model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
        model = model.to(device)
        model.eval()
        print(f"ESMFold loaded on {device}.")
        use_hf = True
    except Exception as e:
        print(f"HuggingFace ESMFold loading notice: {e}. Falling back to ESM Atlas API...")
        use_hf = False

def fold_sequence(seq, qid):
    pdb_path = PDB_DIR / f"{qid}.pdb"
    if pdb_path.exists() and pdb_path.stat().st_size > 0:
        return pdb_path.read_text(encoding="utf-8")

    if use_hf and model is not None:
        with torch.no_grad():
            inputs = tokenizer([seq], return_tensors="pt", add_special_tokens=False)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)
            pdb_str = model.output_to_pdb(outputs)[0]
            pdb_path.write_text(pdb_str, encoding="utf-8")
            return pdb_str
    else:
        req = urllib.request.Request(
            "https://api.esmatlas.com/v1/fold/",
            data=seq.encode("utf-8"),
            headers={"Content-Type": "text/plain"}
        )
        with urllib.request.urlopen(req, timeout=90) as resp:
            pdb_str = resp.read().decode("utf-8")
            pdb_path.write_text(pdb_str, encoding="utf-8")
            return pdb_str

print(f"Processing 3D structures and extracting pLDDT for {len(tier1_passed)} candidates...")
t0 = time.time()
plddts = []

for idx, row in enumerate(tier1_passed):
    qid = row["id"]
    seq = row["sequence"]
    pdb_content = fold_sequence(seq, qid)

    # Extract CA B-factor pLDDT
    bfactors = []
    for line in pdb_content.splitlines():
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            try:
                bfactors.append(float(line[60:66].strip()))
            except ValueError:
                pass
    mean_plddt = sum(bfactors) / max(len(bfactors), 1)
    if mean_plddt <= 1.05 and mean_plddt > 0.0:
        mean_plddt *= 100.0  # Normalize 0-1 scale to 0-100
    plddts.append(round(mean_plddt, 2))

    if (idx + 1) % 10 == 0 or (idx + 1) == len(tier1_passed):
        print(f"  [{idx+1:>3}/{len(tier1_passed)}] {qid:<28} len={len(seq)} mean_pLDDT={mean_plddt:.1f} ({time.time() - t0:.1f}s)")

for r, p in zip(tier1_passed, plddts):
    r["mean_plddt"] = p
    r["passed_plddt80"] = p >= 80.0

df_plddt = pd.DataFrame(tier1_passed)
df_plddt.to_csv(WORKSPACE / "esmfold_plddt_summary.tsv", sep="	", index=False)
passed_plddt_df = df_plddt[df_plddt["passed_plddt80"]].copy()

print(f"\nStructure Evaluation Complete in {time.time() - t0:.1f}s.")
print(f"High-Confidence Designs (mean pLDDT >= 80.0): {len(passed_plddt_df)} / {len(df_plddt)} ({100*len(passed_plddt_df)/max(len(df_plddt),1):.1f}%)")


Found uploaded coordinate archive: esmfold_pdbs.zip. Extracting to exp3_module_3_workspace/esmfold_pdbs...
Found uploaded coordinate archive: /content/esmfold_pdbs.zip. Extracting to exp3_module_3_workspace/esmfold_pdbs...
Candidate PDB status in exp3_module_3_workspace/esmfold_pdbs: 140 / 140 models ready.
All 140 candidate PDB structures already exist in exp3_module_3_workspace/esmfold_pdbs!
SKIPPING ESMFold model loading and folding entirely to save time and GPU memory.
Processing 3D structures and extracting pLDDT for 140 candidates...
  [ 10/140] petase_n016_T1.0_seq_18      len=267 mean_pLDDT=88.8 (0.0s)
  [ 20/140] petase_n064_T0.8_seq_21      len=262 mean_pLDDT=90.9 (0.0s)
  [ 30/140] petase_n032_T0.8_seq_8       len=257 mean_pLDDT=92.9 (0.0s)
  [ 40/140] petase_n064_T0.8_seq_1       len=266 mean_pLDDT=89.4 (0.0s)
  [ 50/140] petase_n032_T1.0_seq_1       len=256 mean_pLDDT=87.8 (0.0s)
  [ 60/140] petase_n032_T1.0_seq_28      len=246 mean_pLDDT=86.3 (0.1s)
  [ 70/140] petase_n06

## 6. Tier 3: OpenMM Constrained Local Energy Relaxation
**The Essential Fix for Rotamer Clashes & Fold Preservation with PDBFixer:**
ESMFold PDBs omit the C-terminal $OXT$ atom and hydrogens, causing standard AMBER template matchers to fail.
We use **PDBFixer** to automatically reconstruct missing atoms, terminal oxygen, and protonation states before creating the AMBER14SB/OBC2 system.
Harmonic position restraints are applied to all heavy backbone atoms ($N, C\alpha, C, O$) with $k = 10\text{ kcal/mol}\cdot\text{\AA}^2$, allowing side-chain rotamers (catalytic Ser $O\gamma$, His $N\epsilon2$, Asp $O\delta$, aromatic clamp) to resolve steric strain into physical minima while strictly preserving the global fold ($< 0.5\text{ \AA}$ backbone RMSD).


In [6]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile, ForceField, Simulation, NoCutoff
from openmm import CustomExternalForce, LangevinIntegrator, Platform
from openmm.unit import kelvin, picosecond, kilocalories_per_mole, angstrom
import numpy as np

# Verify GPU platform for OpenMM (NoCutoff + OBC2 is ~20x faster on CUDA than CPU)
REQUIRE_GPU = True  # Set False only if you intentionally want to run slow CPU relaxation
available_platforms = [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]
if "CUDA" in available_platforms:
    omm_platform = Platform.getPlatformByName("CUDA")
elif "OpenCL" in available_platforms:
    omm_platform = Platform.getPlatformByName("OpenCL")
else:
    if REQUIRE_GPU:
        raise RuntimeError(
            f"No GPU OpenMM platform (CUDA/OpenCL) detected (available: {available_platforms}). "
            "Run Steps 1-9 on a GPU runtime, or set REQUIRE_GPU = False to override."
        )
    omm_platform = Platform.getPlatformByName("CPU") if "CPU" in available_platforms else Platform.getPlatformByName("Reference")

print(f"OpenMM platform selected: {omm_platform.getName()} (available: {available_platforms})")
print(f"Initiating OpenMM backbone-constrained relaxation on {len(passed_plddt_df)} high-pLDDT designs...")
relaxation_records = []
t_relax_start = time.time()

forcefield = ForceField("amber14/protein.ff14SB.xml", "implicit/obc2.xml")
backbone_names = {"N", "CA", "C", "O"}
FORCE_RELAX = False  # Set True only to overwrite existing *_relaxed.pdb files

for idx, (_, row) in enumerate(passed_plddt_df.iterrows()):
    qid = row["id"]
    in_pdb_path = PDB_DIR / f"{qid}.pdb"
    out_pdb_path = RELAX_DIR / f"{qid}_relaxed.pdb"

    if out_pdb_path.exists() and out_pdb_path.stat().st_size > 0 and not FORCE_RELAX:
        relaxation_records.append({"id": qid, "relaxation_success": True, "backbone_rmsd": 0.0, "status": "cached"})
        continue

    try:
        # PDBFixer adds the missing C-terminal OXT atom and hydrogens that ESMFold omits
        fixer = PDBFixer(filename=str(in_pdb_path))
        fixer.findMissingResidues()
        fixer.findMissingAtoms()
        fixer.addMissingAtoms()
        fixer.addMissingHydrogens(7.0)

        system = forcefield.createSystem(fixer.topology, nonbondedMethod=NoCutoff, constraints=None)

        # Harmonic restraint on backbone heavy atoms: k = 10 kcal/(mol * A^2)
        k_restraint = 10.0 * kilocalories_per_mole / (angstrom ** 2)
        restraint = CustomExternalForce("0.5 * k * ((x-x0)^2 + (y-y0)^2 + (z-z0)^2)")
        restraint.addGlobalParameter("k", k_restraint)
        restraint.addPerParticleParameter("x0")
        restraint.addPerParticleParameter("y0")
        restraint.addPerParticleParameter("z0")

        initial_bb = []
        for atom in fixer.topology.atoms():
            if atom.name in backbone_names:
                pos = fixer.positions[atom.index]
                restraint.addParticle(atom.index, [pos[0], pos[1], pos[2]])
                initial_bb.append(np.array([pos[0]._value, pos[1]._value, pos[2]._value]))

        system.addForce(restraint)

        integrator = LangevinIntegrator(300 * kelvin, 1.0 / picosecond, 0.002 * picosecond)
        sim = Simulation(fixer.topology, system, integrator, omm_platform)
        sim.context.setPositions(fixer.positions)

        sim.minimizeEnergy(maxIterations=500)

        state = sim.context.getState(getPositions=True)
        final_positions = state.getPositions()

        with open(out_pdb_path, "w", encoding="utf-8") as f:
            PDBFile.writeFile(fixer.topology, final_positions, f)

        final_bb = []
        for atom in fixer.topology.atoms():
            if atom.name in backbone_names:
                pos = final_positions[atom.index]
                final_bb.append(np.array([pos[0]._value, pos[1]._value, pos[2]._value]))

        init_arr = np.array(initial_bb) * 10.0  # nm to A
        fin_arr = np.array(final_bb) * 10.0
        bb_rmsd = float(np.sqrt(np.mean(np.sum((init_arr - fin_arr) ** 2, axis=1))))

        relaxation_records.append({
            "id": qid,
            "relaxation_success": True,
            "backbone_rmsd": round(bb_rmsd, 3),
            "status": "minimized"
        })
    except Exception as e:
        print(f"Relaxation notice for {qid}: {e}. Copying unrelaxed model.")
        shutil.copy(in_pdb_path, out_pdb_path)
        relaxation_records.append({"id": qid, "relaxation_success": False, "backbone_rmsd": 0.0, "status": "failed"})

    if (idx + 1) % 5 == 0 or (idx + 1) == len(passed_plddt_df):
        print(f"  [{idx+1:>3}/{len(passed_plddt_df)}] Relaxed {qid:<28} ({time.time() - t_relax_start:.1f}s)")

df_relax = pd.DataFrame(relaxation_records)
df_relax.to_csv(WORKSPACE / "relaxation_results.tsv", sep="	", index=False)
print(f"\nRelaxation Complete. Successfully minimized models: {df_relax['relaxation_success'].sum()} / {len(df_relax)}")
if not df_relax[df_relax['backbone_rmsd'] > 0].empty:
    print(f"Mean backbone RMSD: {df_relax[df_relax['backbone_rmsd'] > 0]['backbone_rmsd'].mean():.3f} A (Verifies fold preservation)")


Initiating OpenMM backbone-constrained relaxation on 139 high-pLDDT designs...
  [ 15/139] Relaxed petase_n016_T0.8_seq_1       (142.8s)
  [ 30/139] Relaxed petase_n032_T0.8_seq_8       (250.7s)
  [ 45/139] Relaxed petase_n032_T1.0_seq_34      (368.5s)
  [ 60/139] Relaxed petase_n032_T1.0_seq_28      (520.0s)
  [ 75/139] Relaxed petase_n008_T0.8_seq_13      (651.0s)
  [ 90/139] Relaxed petase_n064_T1.0_seq_15      (775.3s)
  [105/139] Relaxed petase_n064_T0.8_seq_29      (894.6s)
  [120/139] Relaxed petase_n064_T0.8_seq_23      (1003.6s)
  [135/139] Relaxed petase_n032_T0.8_seq_10      (1101.7s)
  [139/139] Relaxed petase_n016_T0.8_seq_36      (1133.0s)

Relaxation Complete. Successfully minimized models: 139 / 139
Mean backbone RMSD: 0.116 A (Verifies fold preservation)


## 7. Tier 4: Foldseek Structural Alignment vs IsPETase (6EQE) & LCC (4EB0)
We align all relaxed PDB models against crystallographic reference structures using Foldseek TM-score mode (`--alignment-type 1`).


In [7]:
FOLDSEEK = pathlib.Path("foldseek/bin/foldseek").resolve()
ALIGN_OUT = WORKSPACE / "foldseek_vs_references.m8"
TMP_DIR = WORKSPACE / "tmp_foldseek"
TMP_DIR.mkdir(exist_ok=True)

FIELDS = "query,target,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits,alntmscore,rmsd,lddt"

cmd = [
    str(FOLDSEEK), "easy-search",
    str(RELAX_DIR), str(REFS_DIR),
    str(ALIGN_OUT), str(TMP_DIR),
    "--format-output", FIELDS,
    "-e", "1.0",
    "--alignment-type", "1"  # TM-score alignment mode
]

print("Executing Foldseek structural alignment...")
res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode != 0:
    print("Foldseek stderr:", res.stderr)
else:
    print(f"Foldseek alignment completed! Total alignment rows: {sum(1 for _ in open(ALIGN_OUT))}")

df_fs = pd.read_csv(ALIGN_OUT, sep="\t", names=FIELDS.split(","))
df_fs["query"] = df_fs["query"].str.replace(".pdb", "", regex=False).str.replace("_relaxed", "", regex=False)
df_fs["target"] = df_fs["target"].str.replace(".pdb", "", regex=False)

print("\nTop alignments vs 6EQE (IsPETase):")
print(df_fs[df_fs["target"] == "6EQE"].sort_values(by="alntmscore", ascending=False)[["query", "target", "fident", "alntmscore", "rmsd", "evalue"]].head(10))


Executing Foldseek structural alignment...
Foldseek alignment completed! Total alignment rows: 278

Top alignments vs 6EQE (IsPETase):
                       query target  fident  alntmscore   rmsd  evalue
225  petase_n064_T0.8_seq_40   6EQE   0.481      0.9744  1.159  0.9449
246  petase_n016_T0.8_seq_37   6EQE   0.465      0.9739  1.194  0.9384
193  petase_n064_T1.0_seq_33   6EQE   0.439      0.9734  1.157  0.9345
166   petase_n032_T0.8_seq_4   6EQE   0.562      0.9711  1.172  0.9492
0    petase_n008_T0.8_seq_25   6EQE   0.437      0.9708  1.073  0.9459
262   petase_n016_T0.8_seq_5   6EQE   0.456      0.9707  1.224  0.9487
16   petase_n032_T1.0_seq_18   6EQE   0.471      0.9700  1.150  0.9201
269  petase_n032_T0.8_seq_31   6EQE   0.446      0.9685  1.249  0.9373
101  petase_n032_T1.0_seq_23   6EQE   0.464      0.9682  1.221  0.9366
272  petase_n032_T1.0_seq_19   6EQE   0.513      0.9681  1.307  0.9407


## 8. Tier 5: 3D Active-Site & Groove Architecture Audit vs IsPETase (6EQE)
Superposes each query structure onto `6EQE` using global sequence alignment to prevent N-terminal indexing errors:
- Evaluates catalytic triad internal spatial geometry: $\text{Ser } O\gamma - \text{His } N\epsilon2 \le 3.5\text{ \AA}$ and $\text{His } N\delta1 - \text{Asp } O\delta \le 3.2\text{ \AA}$.
- Preserves native Ser $O\gamma$ coordinates in the receptor disk coordinate frame for docking.
- Maps residue numbers of Ser, His, Asp, and Trp clamp to construct `--flexres` for flexible Smina docking.
- Detects presence of oxyanion donor (Tyr87) and substrate clamp (Trp159 & Trp185).


In [8]:
from Bio.PDB import PDBParser, Superimposer
from Bio import Align
from Bio.Align import substitution_matrices
from Bio.SeqUtils import seq1

parser = PDBParser(QUIET=True)
ref_6eqe_path = REFS_DIR / "6EQE.pdb"
ref_struct = parser.get_structure("6EQE", str(ref_6eqe_path))[0]["A"]

aligner = Align.PairwiseAligner()
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.mode = "global"
aligner.open_gap_score = -10.0
aligner.extend_gap_score = -0.5

def chain_seq_and_res(chain):
    res_list = [r for r in chain if r.id[0] == " " and "CA" in r]
    seq_str = "".join(seq1(r.get_resname()) for r in res_list)
    return seq_str, res_list

ref_seq, ref_res = chain_seq_and_res(ref_struct)

def audit_active_site(pdb_path):
    try:
        qry_struct = parser.get_structure("qry", str(pdb_path))[0]
        qry_chain = list(qry_struct.get_chains())[0]
    except Exception:
        return None

    qry_seq, qry_res = chain_seq_and_res(qry_chain)
    try:
        aln = aligner.align(ref_seq, qry_seq)[0]
    except Exception:
        return None

    ref_ca_pairs, qry_ca_pairs = [], []
    resnum_to_qry = {}
    for (rs, re_), (qs, qe) in zip(*aln.aligned):
        for k in range(re_ - rs):
            rr = ref_res[rs + k]
            qr = qry_res[qs + k]
            ref_ca_pairs.append(rr["CA"])
            qry_ca_pairs.append(qr["CA"])
            resnum_to_qry[rr.id[1]] = qr

    qry_ser = resnum_to_qry.get(160)
    qry_asp = resnum_to_qry.get(206)
    qry_his = resnum_to_qry.get(237)
    qry_w159_equiv = resnum_to_qry.get(159)
    qry_w185_equiv = resnum_to_qry.get(185)

    native_ser_og = (
        qry_ser["OG"].get_coord().copy().tolist()
        if (qry_ser is not None and qry_ser.get_resname() == "SER" and "OG" in qry_ser)
        else None
    )

    seq_triad_intact = (
        qry_ser is not None and qry_ser.get_resname() == "SER" and
        qry_asp is not None and qry_asp.get_resname() == "ASP" and
        qry_his is not None and qry_his.get_resname() == "HIS"
    )

    # Build --flexres string for Smina (keep catalytic Ser rigid so ser_og_coord remains exact;
    # exclude buried Asp to prevent combinatorial search explosion and keep His-Asp H-bond intact)
    flex_res_ids = []
    if qry_his is not None:
        flex_res_ids.append(f"{qry_chain.id}:{qry_his.id[1]}")
    if qry_w159_equiv is not None and qry_w159_equiv.get_resname() in ["TRP", "TYR", "PHE"]:
        flex_res_ids.append(f"{qry_chain.id}:{qry_w159_equiv.id[1]}")
    elif qry_w185_equiv is not None and qry_w185_equiv.get_resname() in ["TRP", "TYR", "PHE"]:
        flex_res_ids.append(f"{qry_chain.id}:{qry_w185_equiv.id[1]}")
    flexres_str = ",".join(flex_res_ids) if flex_res_ids else None

    if not seq_triad_intact or "OG" not in qry_ser or "NE2" not in qry_his:
        return {
            "seq_triad_intact": seq_triad_intact, "triad_intact": False,
            "d_ser_his": 99.0, "d_his_asp": 99.0,
            "has_oxyanion_tyr": False, "has_groove_w159": False, "has_groove_w185": False,
            "active_site_complete": False, "ser_og_coord": native_ser_og,
            "flexres_str": flexres_str
        }

    ser_og = qry_ser["OG"].get_coord()
    his_ne2 = qry_his["NE2"].get_coord()
    his_nd1 = qry_his["ND1"].get_coord() if "ND1" in qry_his else his_ne2
    asp_od = qry_asp["OD1"].get_coord() if "OD1" in qry_asp else (qry_asp["OD2"].get_coord() if "OD2" in qry_asp else None)

    d_ser_his = float(np.linalg.norm(ser_og - his_ne2))
    d_his_asp = float(np.linalg.norm(his_nd1 - asp_od)) if asp_od is not None else 99.0
    triad_intact = (d_ser_his <= 3.5) and (d_his_asp <= 3.2)

    # Superpose to check oxyanion and groove aromatic clamp
    sup = Superimposer()
    sup.set_atoms(ref_ca_pairs, qry_ca_pairs)
    sup.apply(qry_struct.get_atoms())

    ref_y87_ca = ref_struct[87]["CA"].get_coord()
    ref_w159_ca = ref_struct[159]["CA"].get_coord()
    ref_w185_ca = ref_struct[185]["CA"].get_coord()

    qry_residues = [r for r in qry_chain if r.id[0] == " " and "CA" in r]
    has_oxyanion_tyr = any(r.get_resname() in ["TYR", "PHE", "TRP", "HIS"] and np.linalg.norm(r["CA"].get_coord() - ref_y87_ca) < 3.5 for r in qry_residues)
    has_w159 = any(r.get_resname() in ["TRP", "TYR", "PHE"] and np.linalg.norm(r["CA"].get_coord() - ref_w159_ca) < 3.5 for r in qry_residues)
    has_w185 = any(r.get_resname() in ["TRP", "TYR", "PHE"] and np.linalg.norm(r["CA"].get_coord() - ref_w185_ca) < 3.8 for r in qry_residues)

    return {
        "seq_triad_intact": True,
        "triad_intact": triad_intact,
        "d_ser_his": round(d_ser_his, 2),
        "d_his_asp": round(d_his_asp, 2),
        "has_oxyanion_tyr": has_oxyanion_tyr,
        "has_groove_w159": has_w159,
        "has_groove_w185": has_w185,
        "active_site_complete": triad_intact and has_oxyanion_tyr and (has_w159 or has_w185),
        "ser_og_coord": native_ser_og,
        "flexres_str": flexres_str
    }

print("Auditing 3D active-site pocket geometry...")
triad_rows = []
for _, row in passed_plddt_df.iterrows():
    qid = row["id"]
    pdb_path = RELAX_DIR / f"{qid}_relaxed.pdb"
    if not pdb_path.exists():
        pdb_path = PDB_DIR / f"{qid}.pdb"
    res = audit_active_site(pdb_path)
    if res:
        res["id"] = qid
        res["mean_plddt"] = row["mean_plddt"]
        triad_rows.append(res)

df_triads = pd.DataFrame(triad_rows)
df_triads.to_csv(WORKSPACE / "catalytic_triad_geometry_report.tsv", sep="	", index=False)
print(f"Active-Site Pocket Audit Results:")
print(f"  Intact Catalytic Triad (Ser-His <= 3.5A, His-Asp <= 3.2A): {df_triads['triad_intact'].sum()} / {len(df_triads)}")
print(f"  Oxyanion Hole Donor (Tyr87 equivalent) Present:          {df_triads['has_oxyanion_tyr'].sum()} / {len(df_triads)}")
print(f"  Complete Functional Pocket (Triad + Oxyanion + Clamp):   {df_triads['active_site_complete'].sum()} / {len(df_triads)}")
print(f"  Constructed flexible residue targets (--flexres):       {df_triads['flexres_str'].notna().sum()} / {len(df_triads)}")


Auditing 3D active-site pocket geometry...
Active-Site Pocket Audit Results:
  Intact Catalytic Triad (Ser-His <= 3.5A, His-Asp <= 3.2A): 133 / 139
  Oxyanion Hole Donor (Tyr87 equivalent) Present:          132 / 139
  Complete Functional Pocket (Triad + Oxyanion + Clamp):   128 / 139
  Constructed flexible residue targets (--flexres):       139 / 139


## 9. Prepare Docking Ligands: 2-HE(MHET) & 3PET Trimeric PET Fragment
Generates 3D conformations and coordinates for:
1. **2-HE(MHET):** `O=C(O)c1ccc(C(=O)OCCO)cc1` (Monoradical PET degradation intermediate).
2. **3PET Fragment:** `O=C(O)c1ccc(C(=O)OCCOc2ccc(C(=O)OCCOc3ccc(C(=O)OCCO)cc3)cc2)cc1` (Trimeric PET chain representing insoluble plastic surface).
Maps 1-to-1 pairings between each ester carbonyl carbon and its directly bonded carbonyl oxygen.
Outputs SDF format directly consumable by Smina (no Meeko/OpenBabel dependency required).


In [9]:
from rdkit import Chem
from rdkit.Chem import AllChem

ligands_config = {
    "2he_mhet": {
        "smiles": "O=C(O)c1ccc(C(=O)OCCO)cc1",
        "sdf": LIGAND_DIR / "2he_mhet.sdf",
        "pdbqt": LIGAND_DIR / "2he_mhet.pdbqt",
        "ester_smarts": "[CX3](=O)[OX2][CH2]"
    },
    "3pet": {
        "smiles": "O=C(O)c1ccc(C(=O)OCCOc2ccc(C(=O)OCCOc3ccc(C(=O)OCCO)cc3)cc2)cc1",
        "sdf": LIGAND_DIR / "3pet.sdf",
        "pdbqt": LIGAND_DIR / "3pet.pdbqt",
        "ester_smarts": "[CX3](=O)[OX2][CH2]"
    }
}

for name, conf in ligands_config.items():
    mol = Chem.MolFromSmiles(conf["smiles"])
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
    AllChem.MMFFOptimizeMolecule(mol)
    Chem.MolToMolFile(mol, str(conf["sdf"]))

    # Match each ester carbonyl carbon (m[0]) with its bonded carbonyl oxygen (m[1])
    patt = Chem.MolFromSmarts(conf["ester_smarts"])
    matches = mol.GetSubstructMatches(patt)
    conf["ester_c_indices"] = [m[0] for m in matches]
    conf["carbonyl_o_indices"] = [m[1] for m in matches]

    # Convert to PDBQT as optional format (support both meeko versions & obabel)
    has_meeko = False
    try:
        import meeko
        mol_clean = Chem.MolFromMolFile(str(conf["sdf"]), removeHs=False)
        prep = meeko.MoleculePreparation()
        setups = prep.prepare(mol_clean)
        for s in setups:
            if hasattr(s, "write_pdbqt_string"):
                conf["pdbqt"].write_text(s.write_pdbqt_string())
                has_meeko = True
                break
            elif hasattr(meeko, "PDBQTWriterLegacy"):
                pdbqt_str, is_ok, _ = meeko.PDBQTWriterLegacy.write_string(s)
                if is_ok:
                    conf["pdbqt"].write_text(pdbqt_str)
                    has_meeko = True
                    break
    except Exception as e:
        pass

    if not has_meeko and not conf["pdbqt"].exists():
        obabel_exe = shutil.which("obabel")
        if obabel_exe:
            subprocess.run([obabel_exe, str(conf["sdf"]), "-O", str(conf["pdbqt"]), "-h"], capture_output=True)

    print(f"Ligand {name:<8} SDF ready: {conf['sdf'].exists()} | PDBQT ready: {conf['pdbqt'].exists()}")
    print(f"  Paired Ester C & Carbonyl O indices: {list(zip(conf['ester_c_indices'], conf['carbonyl_o_indices']))}")


Ligand 2he_mhet SDF ready: True | PDBQT ready: True
  Paired Ester C & Carbonyl O indices: [(7, 8)]
Ligand 3pet     SDF ready: True | PDBQT ready: True
  Paired Ester C & Carbonyl O indices: [(7, 8), (17, 18), (27, 28)]


## 9b. Export GPU Phase Checkpoint (Stop GPU Instance Here)
Saves all GPU-phase artifacts (`relaxed_pdbs/`, `esmfold_pdbs/`, `docking_ligands/`, `references/`, `ligands_config.json`, and all TSV/Foldseek reports) into `exp3_module_3_gpu_checkpoint.zip` and syncs to Google Drive.
After running this cell, you can safely shut down the GPU runtime and switch to a CPU instance to run **Step 10a + Steps 10–13**.


In [ ]:
import json, shutil

# Save ligand config metadata so Step 10 on CPU can restore without re-running Step 9
lig_serializable = {}
for k, v in ligands_config.items():
    lig_serializable[k] = {
        "smiles": v["smiles"],
        "sdf": str(v["sdf"]),
        "pdbqt": str(v["pdbqt"]),
        "ester_smarts": v["ester_smarts"],
        "ester_c_indices": [int(x) for x in v.get("ester_c_indices", [])],
        "carbonyl_o_indices": [int(x) for x in v.get("carbonyl_o_indices", [])],
    }
(WORKSPACE / "ligands_config.json").write_text(json.dumps(lig_serializable, indent=2), encoding="utf-8")

print("Creating GPU phase checkpoint archive (exp3_module_3_gpu_checkpoint.zip)...")
shutil.make_archive("exp3_module_3_gpu_checkpoint", "zip", WORKSPACE)
shutil.make_archive("relaxed_pdbs", "zip", RELAX_DIR)
ckpt_size_mb = pathlib.Path("exp3_module_3_gpu_checkpoint.zip").stat().st_size / (1024 * 1024)
print(f"Created exp3_module_3_gpu_checkpoint.zip ({ckpt_size_mb:.1f} MB)")

if "google.colab" in sys.modules and pathlib.Path("/content/drive/MyDrive").exists():
    drive_out = pathlib.Path("/content/drive/MyDrive/experiment_3/module_3_results")
    drive_out.mkdir(parents=True, exist_ok=True)
    for fname in ["exp3_module_3_gpu_checkpoint.zip", "relaxed_pdbs.zip"]:
        if pathlib.Path(fname).exists():
            shutil.copy(fname, drive_out / fname)
    for f in list(WORKSPACE.glob("*.tsv")) + list(WORKSPACE.glob("*.m8")) + [WORKSPACE / "ligands_config.json"]:
        if f.exists():
            shutil.copy(f, drive_out / f.name)
    print(f"Synced GPU checkpoint to Google Drive: {drive_out}")
else:
    print("Note: Google Drive not mounted; download exp3_module_3_gpu_checkpoint.zip manually if moving to another machine.")

print("\n" + "=" * 70)
print("GPU PHASE (Steps 1-9) COMPLETE — READY FOR CPU INSTANCE AT STEP 10a")
print("=" * 70)


## 10a. CPU Instance Setup & Restore GPU Checkpoint (Start Here on CPU Instance)
When switching to a **CPU-only instance** for Steps 10–13, you do **not** need to run Steps 1–9. Run this single cell first:
1. Mounts Google Drive (if in Colab) and initializes workspace paths.
2. Installs only the lightweight CPU docking & plotting tools (`smina`, `openbabel`, `rdkit`, `pandas`, `matplotlib`, `seaborn`) — skipping PyTorch, Transformers, ESMFold, OpenMM, PDBFixer, and Foldseek.
3. Restores `exp3_module_3_gpu_checkpoint.zip` (from local folder or Google Drive) into `exp3_module_3_workspace/` and loads `df_triads` & `ligands_config`.


In [ ]:
import ast, importlib, json, os, pathlib, shutil, subprocess, sys, time, zipfile

# 1. Mount Google Drive if running in Colab
if "google.colab" in sys.modules and not pathlib.Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

# 2. Initialize workspace directories
WORKSPACE = pathlib.Path("exp3_module_3_workspace")
PDB_DIR = WORKSPACE / "esmfold_pdbs"
RELAX_DIR = WORKSPACE / "relaxed_pdbs"
DOCK_DIR = WORKSPACE / "docking_results"
LIGAND_DIR = WORKSPACE / "docking_ligands"
REFS_DIR = WORKSPACE / "references"
RECEPTOR_PDBQT_DIR = WORKSPACE / "receptor_pdbqts"

for d in [WORKSPACE, PDB_DIR, RELAX_DIR, DOCK_DIR, LIGAND_DIR, REFS_DIR, RECEPTOR_PDBQT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 3. Install lightweight CPU docking & analysis dependencies only if missing
if not shutil.which("obabel"):
    print("Installing openbabel...")
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "openbabel", "curl"], check=False)

missing_pkgs = []
for mod, pkg in [("rdkit", "rdkit"), ("pandas", "pandas"), ("numpy", "numpy"), ("matplotlib", "matplotlib"), ("seaborn", "seaborn")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing_pkgs.append(pkg)
if missing_pkgs:
    print(f"Installing Python packages: {missing_pkgs}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs], check=True)

smina_exe = shutil.which("smina") or "/usr/local/bin/smina"
if not pathlib.Path(smina_exe).exists():
    print("Downloading static Smina binary...")
    if not shutil.which("curl"):
        subprocess.run(["apt-get", "update", "-qq"], check=False)
        subprocess.run(["apt-get", "install", "-y", "-qq", "curl"], check=False)
    subprocess.run(
        ["curl", "-fsSL", "https://sourceforge.net/projects/smina/files/smina.static/download", "-o", "/usr/local/bin/smina"],
        check=True,
    )
    subprocess.run(["chmod", "+x", "/usr/local/bin/smina"], check=True)

import numpy as np
import pandas as pd

# 4. Restore GPU checkpoint if workspace is empty
triad_tsv = WORKSPACE / "catalytic_triad_geometry_report.tsv"
if not triad_tsv.exists() or len(list(RELAX_DIR.glob("*_relaxed.pdb"))) == 0:
    search_roots = [
        pathlib.Path("."),
        pathlib.Path("/content"),
        pathlib.Path("/content/drive/MyDrive/experiment_3/module_3_results"),
    ]
    ckpt_zip = None
    for root in search_roots:
        candidate = root / "exp3_module_3_gpu_checkpoint.zip"
        if candidate.exists():
            ckpt_zip = candidate
            break
    if ckpt_zip is None and pathlib.Path("/content/drive").exists():
        matches = list(pathlib.Path("/content/drive").glob("**/exp3_module_3_gpu_checkpoint.zip"))
        if matches:
            ckpt_zip = matches[0]

    if ckpt_zip is not None:
        print(f"Extracting GPU checkpoint from {ckpt_zip} into {WORKSPACE}...")
        with zipfile.ZipFile(ckpt_zip, "r") as zf:
            zf.extractall(WORKSPACE)
    else:
        # Fallback: look for relaxed_pdbs.zip + TSVs in Drive/current dir
        drive_dir = pathlib.Path("/content/drive/MyDrive/experiment_3/module_3_results")
        for root in [pathlib.Path("."), drive_dir]:
            rzip = root / "relaxed_pdbs.zip"
            if rzip.exists() and len(list(RELAX_DIR.glob("*_relaxed.pdb"))) == 0:
                print(f"Extracting {rzip} into {RELAX_DIR}...")
                with zipfile.ZipFile(rzip, "r") as zf:
                    zf.extractall(RELAX_DIR)
            for ext in ("*.tsv", "*.m8", "ligands_config.json"):
                for f in root.glob(ext):
                    dest = WORKSPACE / f.name
                    if not dest.exists():
                        shutil.copy(f, dest)

# 5. Load state into memory for Steps 10-13
if not triad_tsv.exists():
    raise FileNotFoundError(
        "Could not find exp3_module_3_gpu_checkpoint.zip or catalytic_triad_geometry_report.tsv. "
        "Upload exp3_module_3_gpu_checkpoint.zip or mount Google Drive with your GPU phase outputs."
    )

def _parse_coord(val):
    if isinstance(val, (list, tuple, np.ndarray)):
        return [float(x) for x in val]
    if pd.isna(val) or not str(val).strip():
        return None
    return [float(x) for x in ast.literal_eval(str(val))]

df_triads = pd.read_csv(triad_tsv, sep="\t")
df_triads["ser_og_coord"] = df_triads["ser_og_coord"].apply(_parse_coord)

if (WORKSPACE / "esmfold_plddt_summary.tsv").exists():
    df_plddt = pd.read_csv(WORKSPACE / "esmfold_plddt_summary.tsv", sep="\t")
    passed_plddt_df = df_plddt[df_plddt["passed_plddt80"].astype(str).str.lower() == "true"].copy()

lig_json = WORKSPACE / "ligands_config.json"
if lig_json.exists():
    raw_lig = json.loads(lig_json.read_text(encoding="utf-8"))
    ligands_config = {}
    for k, v in raw_lig.items():
        entry = dict(v)
        entry["sdf"] = LIGAND_DIR / pathlib.Path(entry["sdf"]).name
        entry["pdbqt"] = LIGAND_DIR / pathlib.Path(entry["pdbqt"]).name
        ligands_config[k] = entry
else:
    from rdkit import Chem
    from rdkit.Chem import AllChem
    ligands_config = {
        "2he_mhet": {
            "smiles": "O=C(O)c1ccc(C(=O)OCCO)cc1",
            "sdf": LIGAND_DIR / "2he_mhet.sdf",
            "pdbqt": LIGAND_DIR / "2he_mhet.pdbqt",
            "ester_smarts": "[CX3](=O)[OX2][CH2]",
        },
        "3pet": {
            "smiles": "O=C(O)c1ccc(C(=O)OCCOc2ccc(C(=O)OCCOc3ccc(C(=O)OCCO)cc3)cc2)cc1",
            "sdf": LIGAND_DIR / "3pet.sdf",
            "pdbqt": LIGAND_DIR / "3pet.pdbqt",
            "ester_smarts": "[CX3](=O)[OX2][CH2]",
        },
    }
    for _lname, _lconf in ligands_config.items():
        if not _lconf["sdf"].exists():
            _mol = Chem.AddHs(Chem.MolFromSmiles(_lconf["smiles"]))
            AllChem.EmbedMolecule(_mol, AllChem.ETKDGv3())
            AllChem.MMFFOptimizeMolecule(_mol)
            Chem.MolToMolFile(_mol, str(_lconf["sdf"]))

n_relaxed = len(list(RELAX_DIR.glob("*_relaxed.pdb")))
print(f"CPU Docking Environment Ready ({max(1, os.cpu_count() or 2)} CPU threads).")
print(f"  Relaxed PDBs available: {n_relaxed}")
print(f"  Active-site triad rows: {len(df_triads)}")
print(f"  Ligands ready: { {k: v['sdf'].exists() for k, v in ligands_config.items()} if 'ligands_config' in globals() else 'No ligands_config' }")


## 10. Tier 6: Two-Stage Induced-Fit Smina Docking (--flexres) & Multi-Mode NAC
Optimized for a **2-vCPU (1 physical core)** CPU instance using a two-stage screening workflow on OpenMM-relaxed receptors:
1. **Active-Site Pre-Filter (`FILTER_COMPLETE_POCKETS = True`):** Skips designs that already failed Step 8's 3D pocket audit (`triad_intact` + oxyanion donor + Trp clamp).
2. **Stage A — Fast Rigid `2-HE(MHET)` Screening (`exhaustiveness=4`):** Docks against the already OpenMM-relaxed side chains (~2–5 s/design).
3. **Stage B — Targeted `--flexres` Rescue (`TWO_STAGE_FLEX = True`):** Runs flexible His+Trp docking (`exhaustiveness=6`, `timeout_s=120`) only on promising near-pocket designs (`affinity <= -4.5`, `attack_dist <= 5.0` Å) that haven't already achieved strict NAC rigidly.
4. **Gated `3PET` Trimer Docking (`GATE_3PET_ON_2HE = True`):** Docks the 21-torsion `3PET` fragment (`exhaustiveness=4`, rigid on OpenMM-relaxed receptor by default) only on `2-HE(MHET)` NAC/high-affinity candidates.
5. **Resume Caching & Incremental Saves:** Caches `_rigid.sdf` and `_docked.sdf` outputs and saves `smina_flexible_docking_results.tsv` every 5 designs.

Evaluates all 5 modes within $\le 1.5\text{ kcal/mol}$ of the top mode against strict NAC criteria:
$$d_{\text{attack}} \le 3.5\text{ \AA} \quad \text{and} \quad \theta_{\text{Bürgi-Dunitz}} = \angle(O\gamma \cdots C_{\text{ester}} = O_{\text{carbonyl}}) \in [95^\circ, 115^\circ] \quad \text{and} \quad \Delta G_{\text{bind}} \le -5.0\text{ kcal/mol}$$


In [ ]:
import ast
import json
import math
import os
import pathlib
import shutil
import subprocess
import time
import numpy as np
import pandas as pd
from rdkit import Chem

# --- Speed / Accuracy Configuration (tuned for 2-vCPU / 1-core CPU instance) ---
FILTER_COMPLETE_POCKETS = True   # Skip designs missing intact triad / oxyanion / Trp clamp in Step 8
TWO_STAGE_FLEX = True            # True: fast rigid 2-HE screen first, then --flexres only on near-NAC misses
RIGID_EXHAUSTIVENESS = 2         # Matches 2 CPU threads (~20-25s/design for 2-HE; use 4 for ~45s/design)
FLEX_EXHAUSTIVENESS = 4          # Exhaustiveness for targeted --flexres rescue
FLEX_TIMEOUT_S = 360             # Give 1-core CPU enough time for --flexres (avoid premature kills)
GATE_3PET_ON_2HE = True          # Only dock 21-torsion 3PET on 2-HE(MHET) NAC or near-NAC hits
THREEPET_EXHAUSTIVENESS = 2      # Matches 2 CPU threads for 21-torsion 3PET (~90-150s instead of >240s)
THREEPET_TIMEOUT_S = 480         # Prevent premature 3PET timeout on 1-core CPU
FLEX_3PET = False                # Keep 3PET rigid on OpenMM-relaxed receptor
SAVE_EVERY_N = 5                 # Save TSV checkpoint every N designs
# ------------------------------------------------------------------------

if "WORKSPACE" not in globals():
    WORKSPACE = pathlib.Path("exp3_module_3_workspace")
    PDB_DIR = WORKSPACE / "esmfold_pdbs"
    RELAX_DIR = WORKSPACE / "relaxed_pdbs"
    DOCK_DIR = WORKSPACE / "docking_results"
    LIGAND_DIR = WORKSPACE / "docking_ligands"
    REFS_DIR = WORKSPACE / "references"
    for _d in [WORKSPACE, PDB_DIR, RELAX_DIR, DOCK_DIR, LIGAND_DIR, REFS_DIR]:
        _d.mkdir(parents=True, exist_ok=True)

def _parse_coord(val):
    if isinstance(val, (list, tuple, np.ndarray)):
        return [float(x) for x in val]
    if pd.isna(val) or not str(val).strip():
        return None
    return [float(x) for x in ast.literal_eval(str(val))]

if "df_triads" not in globals() or df_triads is None:
    triad_tsv = WORKSPACE / "catalytic_triad_geometry_report.tsv"
    if not triad_tsv.exists():
        raise FileNotFoundError(
            f"Missing {triad_tsv}. Run Step 10a to restore the GPU checkpoint before running Step 10."
        )
    df_triads = pd.read_csv(triad_tsv, sep="\t")
    df_triads["ser_og_coord"] = df_triads["ser_og_coord"].apply(_parse_coord)
    print(f"Loaded {len(df_triads)} active-site audit rows from {triad_tsv}.")

if "ligands_config" not in globals() or ligands_config is None:
    lig_json = WORKSPACE / "ligands_config.json"
    if lig_json.exists():
        raw_lig = json.loads(lig_json.read_text(encoding="utf-8"))
        ligands_config = {}
        for k, v in raw_lig.items():
            entry = dict(v)
            entry["sdf"] = pathlib.Path(entry["sdf"])
            if not entry["sdf"].exists():
                entry["sdf"] = LIGAND_DIR / entry["sdf"].name
            entry["pdbqt"] = pathlib.Path(entry["pdbqt"])
            if not entry["pdbqt"].exists():
                entry["pdbqt"] = LIGAND_DIR / entry["pdbqt"].name
            ligands_config[k] = entry
    else:
        ligands_config = {
            "2he_mhet": {
                "smiles": "O=C(O)c1ccc(C(=O)OCCO)cc1",
                "sdf": LIGAND_DIR / "2he_mhet.sdf",
                "pdbqt": LIGAND_DIR / "2he_mhet.pdbqt",
                "ester_smarts": "[CX3](=O)[OX2][CH2]",
            },
            "3pet": {
                "smiles": "O=C(O)c1ccc(C(=O)OCCOc2ccc(C(=O)OCCOc3ccc(C(=O)OCCO)cc3)cc2)cc1",
                "sdf": LIGAND_DIR / "3pet.sdf",
                "pdbqt": LIGAND_DIR / "3pet.pdbqt",
                "ester_smarts": "[CX3](=O)[OX2][CH2]",
            },
        }
    print(f"Loaded ligand configs from disk: {list(ligands_config.keys())}")

RECEPTOR_PDBQT_DIR = WORKSPACE / "receptor_pdbqts"
RECEPTOR_PDBQT_DIR.mkdir(exist_ok=True)
SMINA_EXE = shutil.which("smina") or "/usr/local/bin/smina"
if not pathlib.Path(SMINA_EXE).exists():
    raise FileNotFoundError(f"Smina binary not found at {SMINA_EXE}. Please run Step 10a or Step 2 first.")
OBABEL_EXE = shutil.which("obabel")
N_CPUS = max(1, os.cpu_count() or 2)

def prepare_receptor_pdbqt(pdb_path):
    """Convert all-atom OpenMM PDB to united-atom PDBQT once (merges nonpolar H, preserves chain & resnums)."""
    pdbqt_path = RECEPTOR_PDBQT_DIR / f"{pdb_path.stem}.pdbqt"
    if pdbqt_path.exists() and pdbqt_path.stat().st_size > 0:
        return pdbqt_path
    if OBABEL_EXE:
        res = subprocess.run(
            [OBABEL_EXE, str(pdb_path), "-O", str(pdbqt_path), "-xr"],
            capture_output=True, text=True
        )
        if res.returncode == 0 and pdbqt_path.exists() and pdbqt_path.stat().st_size > 0:
            return pdbqt_path
    return pdb_path

def select_flexres_for_ligand(flexres_str, lig_name):
    """Trim --flexres to avoid combinatorial timeouts:
    - If an older 3-residue string (His,Asp,Trp) is passed, drop buried Asp.
    - Use His+Trp (2 residues) for 2-HE(MHET) and Trp clamp (1 residue) for 21-torsion 3PET.
    """
    if not flexres_str or (isinstance(flexres_str, float) and np.isnan(flexres_str)):
        return None
    parts = [p.strip() for p in str(flexres_str).split(",") if p.strip()]
    if not parts:
        return None
    if len(parts) >= 3:
        parts = [parts[0], parts[-1]]
    if lig_name == "3pet":
        return parts[-1]
    return ",".join(parts[:2])

def calc_bd_angle(ser_og, ester_c, carbonyl_o):
    v_c_to_ser = ser_og - ester_c
    v_c_to_o = carbonyl_o - ester_c
    n_ser = np.linalg.norm(v_c_to_ser)
    n_o = np.linalg.norm(v_c_to_o)
    if n_ser < 1e-6 or n_o < 1e-6:
        return 0.0
    cos_th = np.clip(np.dot(v_c_to_ser, v_c_to_o) / (n_ser * n_o), -1.0, 1.0)
    return round(math.degrees(math.acos(cos_th)), 2)

DEFAULT_DOCK_RES = {
    "affinity": 0.0, "attack_dist": 99.0, "burgi_dunitz_angle": 0.0,
    "nac_score": 0.0, "is_nac": False, "clash": False, "best_mode": 1
}

def parse_all_docking_modes_sdf(out_sdf_path, ser_og, ester_smarts, max_energy_diff=1.5):
    if not out_sdf_path.exists() or out_sdf_path.stat().st_size == 0:
        return dict(DEFAULT_DOCK_RES)

    patt = Chem.MolFromSmarts(ester_smarts)
    suppl = Chem.SDMolSupplier(str(out_sdf_path), removeHs=False)
    mols = [m for m in suppl if m is not None]
    if not mols:
        return dict(DEFAULT_DOCK_RES)

    def get_aff(mol):
        for prop in ("minimizedAffinity", "score", "vina_affinity"):
            if mol.HasProp(prop):
                try:
                    return float(mol.GetProp(prop))
                except ValueError:
                    pass
        return 0.0

    top_aff = get_aff(mols[0])
    evaluated = []

    for mode_idx, mol in enumerate(mols, start=1):
        aff = get_aff(mol)
        conf = mol.GetConformer()
        matches = mol.GetSubstructMatches(patt)

        min_d = 99.0
        best_ang = 0.0
        for m in matches:
            c_idx, o_idx = m[0], m[1]
            c_pos = np.array(conf.GetAtomPosition(c_idx))
            o_pos = np.array(conf.GetAtomPosition(o_idx))
            d = float(np.linalg.norm(c_pos - ser_og))
            if d < min_d:
                min_d = d
                best_ang = calc_bd_angle(ser_og, c_pos, o_pos)

        attack_dist = round(min_d, 2)
        clash = attack_dist < 2.2
        if attack_dist < 2.5:
            nac_score = round(1.0 / (1.0 + (2.5 - attack_dist) * 2.0), 3)
        else:
            nac_score = round(1.0 / (1.0 + max(0.0, attack_dist - 3.2)), 3)

        is_nac = (2.5 <= attack_dist <= 3.5) and (95.0 <= best_ang <= 115.0) and (aff <= -5.0)
        within_window = (aff <= top_aff + max_energy_diff)

        evaluated.append({
            "mode": mode_idx,
            "affinity": aff,
            "attack_dist": attack_dist,
            "burgi_dunitz_angle": best_ang,
            "nac_score": nac_score,
            "is_nac": is_nac,
            "clash": clash,
            "within_window": within_window
        })

    nac_modes = [em for em in evaluated if em["is_nac"] and em["within_window"]]
    if nac_modes:
        best = sorted(nac_modes, key=lambda x: (x["attack_dist"], -x["nac_score"]))[0]
    else:
        window_modes = [em for em in evaluated if em["within_window"]]
        best = sorted(window_modes, key=lambda x: x["attack_dist"])[0] if window_modes else evaluated[0]

    return {
        "affinity": best["affinity"],
        "attack_dist": best["attack_dist"],
        "burgi_dunitz_angle": best["burgi_dunitz_angle"],
        "nac_score": best["nac_score"],
        "is_nac": best["is_nac"],
        "clash": best["clash"],
        "best_mode": best["mode"]
    }

def run_smina_job(receptor_path, ser_coord, flexres_str, lig_conf, out_sdf,
                  box_size=18.0, exhaustiveness=4, timeout_s=120, fallback_rigid=True):
    ser_og = np.array(ser_coord)

    if out_sdf.exists() and out_sdf.stat().st_size > 0:
        return parse_all_docking_modes_sdf(
            out_sdf, ser_og, lig_conf["ester_smarts"], max_energy_diff=1.5
        )

    def build_cmd(flex_arg, exh):
        c = [
            str(SMINA_EXE),
            "-r", str(receptor_path),
            "-l", str(lig_conf["sdf"]),
            "--center_x", f"{ser_coord[0]:.2f}",
            "--center_y", f"{ser_coord[1]:.2f}",
            "--center_z", f"{ser_coord[2]:.2f}",
            "--size_x", f"{box_size:.1f}", "--size_y", f"{box_size:.1f}", "--size_z", f"{box_size:.1f}",
            "--exhaustiveness", str(exh),
            "--num_modes", "5",
            "--cpu", str(N_CPUS),
            "--seed", "42",
            "-o", str(out_sdf)
        ]
        if flex_arg:
            c.extend(["--flexres", flex_arg])
        return c

    ran_ok = False
    try:
        res = subprocess.run(build_cmd(flexres_str, exhaustiveness), capture_output=True, text=True, timeout=timeout_s)
        if res.returncode == 0 and out_sdf.exists() and out_sdf.stat().st_size > 0:
            ran_ok = True
        elif res.returncode != 0:
            print(f"Smina warning ({out_sdf.name}): {res.stderr.strip()[:200]}")
    except subprocess.TimeoutExpired:
        print(f"Smina timeout after {timeout_s}s ({out_sdf.name})")
        out_sdf.unlink(missing_ok=True)
    except Exception as e:
        print(f"Smina exception ({out_sdf.name}): {e}")

    if not ran_ok and flexres_str and fallback_rigid:
        out_sdf.unlink(missing_ok=True)
        try:
            res_fb = subprocess.run(build_cmd(None, RIGID_EXHAUSTIVENESS), capture_output=True, text=True, timeout=90)
            if res_fb.returncode != 0:
                out_sdf.unlink(missing_ok=True)
        except Exception:
            out_sdf.unlink(missing_ok=True)

    return parse_all_docking_modes_sdf(
        out_sdf, ser_og, lig_conf["ester_smarts"], max_energy_diff=1.5
    )

# Filter targets based on Step 8 active-site audit
dock_targets = df_triads[df_triads["ser_og_coord"].notna()].copy()
if FILTER_COMPLETE_POCKETS and "active_site_complete" in dock_targets.columns:
    dock_targets = dock_targets[dock_targets["active_site_complete"].astype(str).str.lower() == "true"].copy()
elif "triad_intact" in dock_targets.columns:
    dock_targets = dock_targets[dock_targets["triad_intact"].astype(str).str.lower() == "true"].copy()

print(
    f"Running Smina docking on {N_CPUS} CPU thread(s) across {len(dock_targets)} / {len(df_triads)} filtered designs "
    f"(TWO_STAGE_FLEX={TWO_STAGE_FLEX}, RIGID_EXH={RIGID_EXHAUSTIVENESS}, FLEX_EXH={FLEX_EXHAUSTIVENESS}, "
    f"GATE_3PET_ON_2HE={GATE_3PET_ON_2HE}, 3PET_EXH={THREEPET_EXHAUSTIVENESS}, OBABEL={'OK' if OBABEL_EXE else 'MISSING'})..."
)

docking_results = []
t_dock_start = time.time()
n_2he_flex_ran = 0
n_3pet_ran = 0

for idx, (_, row) in enumerate(dock_targets.iterrows()):
    qid = row["id"]
    ser_coord = row["ser_og_coord"]
    flexres_raw = row.get("flexres_str")

    pdb_path = RELAX_DIR / f"{qid}_relaxed.pdb"
    if not pdb_path.exists():
        pdb_path = PDB_DIR / f"{qid}.pdb"
    receptor_path = prepare_receptor_pdbqt(pdb_path)
    if idx == 0:
        print(f"  Receptor format verified: {receptor_path.name} (suffix={receptor_path.suffix})")

    t_one = time.time()
    entry = {"id": qid}

    # --- 1. 2-HE(MHET) Docking ---
    lig_2he = ligands_config["2he_mhet"]
    flex_2he = select_flexres_for_ligand(flexres_raw, "2he_mhet")
    out_2he_flex = DOCK_DIR / f"{qid}_2he_mhet_docked.sdf"

    if TWO_STAGE_FLEX:
        out_2he_rigid = DOCK_DIR / f"{qid}_2he_mhet_rigid.sdf"
        res_2he = run_smina_job(
            receptor_path, ser_coord, None, lig_2he, out_2he_rigid,
            box_size=18.0, exhaustiveness=RIGID_EXHAUSTIVENESS, timeout_s=180, fallback_rigid=False
        )
        # Run --flexres rescue on near-NAC misses (affinity <= -5.0, d <= 4.2 A)
        should_flex = (
            (not res_2he["is_nac"])
            and (res_2he["affinity"] <= -5.0)
            and (res_2he["attack_dist"] <= 4.2)
            and bool(flex_2he)
        )
        if should_flex or (out_2he_flex.exists() and out_2he_flex.stat().st_size > 0):
            n_2he_flex_ran += 1
            res_flex = run_smina_job(
                receptor_path, ser_coord, flex_2he, lig_2he, out_2he_flex,
                box_size=18.0, exhaustiveness=FLEX_EXHAUSTIVENESS, timeout_s=FLEX_TIMEOUT_S, fallback_rigid=False
            )
            if res_flex["is_nac"] or (
                not res_2he["is_nac"] and res_flex["affinity"] <= -5.0 and res_flex["attack_dist"] < res_2he["attack_dist"]
            ):
                res_2he = res_flex
    else:
        n_2he_flex_ran += 1
        res_2he = run_smina_job(
            receptor_path, ser_coord, flex_2he, lig_2he, out_2he_flex,
            box_size=18.0, exhaustiveness=FLEX_EXHAUSTIVENESS, timeout_s=FLEX_TIMEOUT_S, fallback_rigid=True
        )

    for k, v in res_2he.items():
        entry[f"2he_mhet_{k}"] = v

    # --- 2. 3PET Trimer Docking (gated or full) ---
    lig_3pet = ligands_config["3pet"]
    out_3pet = DOCK_DIR / f"{qid}_3pet_docked.sdf"
    should_run_3pet = (
        (not GATE_3PET_ON_2HE)
        or res_2he["is_nac"]
        or (res_2he["affinity"] <= -5.0 and res_2he["attack_dist"] <= 3.8)
        or (out_3pet.exists() and out_3pet.stat().st_size > 0)
    )
    if should_run_3pet:
        n_3pet_ran += 1
        flex_3pet = select_flexres_for_ligand(flexres_raw, "3pet") if FLEX_3PET else None
        res_3pet = run_smina_job(
            receptor_path, ser_coord, flex_3pet, lig_3pet, out_3pet,
            box_size=20.0, exhaustiveness=THREEPET_EXHAUSTIVENESS, timeout_s=THREEPET_TIMEOUT_S, fallback_rigid=True
        )
    else:
        res_3pet = dict(DEFAULT_DOCK_RES)

    for k, v in res_3pet.items():
        entry[f"3pet_{k}"] = v

    docking_results.append(entry)
    dt = time.time() - t_one
    print(
        f"  [{idx+1:>3}/{len(dock_targets)}] {qid:<26} ({dt:.1f}s | total {time.time()-t_dock_start:.0f}s) "
        f"2HE: aff={entry['2he_mhet_affinity']:.2f}, d={entry['2he_mhet_attack_dist']:.2f}A, NAC={entry['2he_mhet_is_nac']} | "
        f"3PET: aff={entry['3pet_affinity']:.2f}, d={entry['3pet_attack_dist']:.2f}A"
    )
    if (idx + 1) % SAVE_EVERY_N == 0 or (idx + 1) == len(dock_targets):
        pd.DataFrame(docking_results).to_csv(WORKSPACE / "smina_flexible_docking_results.tsv", sep="\t", index=False)

df_docking = pd.DataFrame(docking_results)
df_docking.to_csv(WORKSPACE / "smina_flexible_docking_results.tsv", sep="\t", index=False)
print(f"\nDocking Complete in {time.time()-t_dock_start:.1f}s (2-HE flex runs: {n_2he_flex_ran}, 3PET runs: {n_3pet_ran}).")
print(f"  2-HE(MHET) Passing Strict NAC: {df_docking['2he_mhet_is_nac'].sum()} / {len(df_docking)}")
print(f"  3PET Trimer Passing Strict NAC: {df_docking['3pet_is_nac'].sum()} / {len(df_docking)}")


## 11. Tier 7: Disulfide Bond Topology & Interfacial Biophysical Proxies
Scans conserved C-terminal disulfides (thermophilic stability determinant) and active-site pairs (DS1/DS2).


In [ ]:
import pathlib, numpy as np, pandas as pd

if "WORKSPACE" not in globals():
    WORKSPACE = pathlib.Path("exp3_module_3_workspace")
    PDB_DIR = WORKSPACE / "esmfold_pdbs"
    RELAX_DIR = WORKSPACE / "relaxed_pdbs"

if "passed_plddt_df" not in globals() or passed_plddt_df is None:
    plddt_tsv = WORKSPACE / "esmfold_plddt_summary.tsv"
    if not plddt_tsv.exists():
        raise FileNotFoundError(f"Missing {plddt_tsv}. Restore GPU checkpoint first (Step 10a).")
    df_plddt = pd.read_csv(plddt_tsv, sep="\t")
    passed_plddt_df = df_plddt[df_plddt["passed_plddt80"].astype(str).str.lower() == "true"].copy()
    print(f"Loaded {len(passed_plddt_df)} high-pLDDT designs from {plddt_tsv}.")

def check_disulfides(pdb_path):
    cys_sg = []
    if not pdb_path.exists():
        return 0, False
    with open(pdb_path, "r") as f:
        for line in f:
            if line.startswith("ATOM") and line[17:20].strip() == "CYS" and line[12:16].strip() == "SG":
                try:
                    resnum = int(line[22:26].strip())
                    x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
                    cys_sg.append((resnum, np.array([x, y, z])))
                except ValueError:
                    pass
    pairs = 0
    has_cterm = False
    for i in range(len(cys_sg)):
        for j in range(i + 1, len(cys_sg)):
            r1, c1 = cys_sg[i]
            r2, c2 = cys_sg[j]
            if np.linalg.norm(c1 - c2) <= 3.0:
                pairs += 1
                if abs(r1 - r2) >= 10:
                    has_cterm = True
    return pairs, has_cterm

ds_records = []
for _, row in passed_plddt_df.iterrows():
    qid = row["id"]
    pdb_path = RELAX_DIR / f"{qid}_relaxed.pdb"
    if not pdb_path.exists():
        pdb_path = PDB_DIR / f"{qid}.pdb"
    count, has_cterm = check_disulfides(pdb_path)
    ds_records.append({"id": qid, "disulfide_count": count, "has_cterm_disulfide": has_cterm})

df_ds = pd.DataFrame(ds_records)
df_ds.to_csv(WORKSPACE / "disulfide_topology_report.tsv", sep="	", index=False)
print(f"Disulfide Topology Audit:")
print(f"  Designs with Intact C-Terminal Disulfide (Tm > 70°C proxy): {df_ds['has_cterm_disulfide'].sum()} / {len(df_ds)}")


## 12. Visualization: pLDDT Distribution, Bürgi-Dunitz NAC Trajectory & Exp 2 Comparison


In [ ]:
import pathlib, pandas as pd
if "WORKSPACE" not in globals():
    WORKSPACE = pathlib.Path("exp3_module_3_workspace")
if "df_plddt" not in globals():
    df_plddt = pd.read_csv(WORKSPACE / "esmfold_plddt_summary.tsv", sep="\t")
if "df_docking" not in globals():
    df_docking = pd.read_csv(WORKSPACE / "smina_flexible_docking_results.tsv", sep="\t")
if "df_ds" not in globals():
    df_ds = pd.read_csv(WORKSPACE / "disulfide_topology_report.tsv", sep="\t")

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(15, 10))

# 1. pLDDT Distribution
plt.subplot(2, 2, 1)
sns.histplot(df_plddt["mean_plddt"], bins=20, kde=True, color="teal")
plt.axvline(80.0, color="red", linestyle="--", label="Filter Threshold (80.0)")
plt.title("ESMFold Confidence (pLDDT) Distribution")
plt.xlabel("Mean pLDDT")
plt.ylabel("Candidate Count")
plt.legend()

# 2. Bürgi-Dunitz Angle vs Attack Distance
plt.subplot(2, 2, 2)
if not df_docking.empty:
    plt.scatter(df_docking["2he_mhet_attack_dist"], df_docking["2he_mhet_burgi_dunitz_angle"],
                c=df_docking["2he_mhet_affinity"], cmap="viridis_r", s=50, alpha=0.8)
    plt.colorbar(label="Binding Affinity (kcal/mol)")
    plt.axvline(3.5, color="red", linestyle="--", label="Max Attack Dist (3.5 A)")
    plt.axhline(95.0, color="orange", linestyle=":", label="NAC Angle Min (95°)")
    plt.axhline(115.0, color="orange", linestyle=":", label="NAC Angle Max (115°)")
    plt.title("2-HE(MHET) Near-Attack Conformation Space")
    plt.xlabel("Attack Distance (A)")
    plt.ylabel("Bürgi-Dunitz Angle (°)")
    plt.legend(loc="upper right")

# 3. Disulfide Counts
plt.subplot(2, 2, 3)
sns.countplot(x="disulfide_count", data=df_ds, palette="Blues_d")
plt.title("Conserved Disulfide Bond Counts")
plt.xlabel("Disulfide Bonds (SG-SG <= 3.0 A)")
plt.ylabel("Candidate Count")

# 4. NAC Yield Comparison: Exp 2 vs Exp 3
plt.subplot(2, 2, 4)
exp2_nac_pct = (1.0 / 70.0) * 100.0  # Exp 2 rigid docking yield: 1.4%
exp3_nac_pct = (df_docking['2he_mhet_is_nac'].sum() / max(len(df_docking), 1)) * 100.0 if not df_docking.empty else 0.0
plt.bar(["Exp 2 (Rigid Smina)", "Exp 3 (Relaxed + Flexible)"], [exp2_nac_pct, exp3_nac_pct], color=["salmon", "seagreen"])
plt.ylabel("NAC Passing Yield (%)")
plt.title("Receptor Flexibility & Relaxation Impact on NAC")

plt.tight_layout()
plt.savefig(WORKSPACE / "module_3_biophysical_summary.png", dpi=300)
plt.show()


## 13. Export Coordinate Archives & Google Drive Sync
Zips tertiary coordinates (`esmfold_pdbs.zip` and `relaxed_pdbs.zip`) for quick local download and copies all artifacts directly into Google Drive.


In [ ]:
import sys, pathlib
if "WORKSPACE" not in globals():
    WORKSPACE = pathlib.Path("exp3_module_3_workspace")
    PDB_DIR = WORKSPACE / "esmfold_pdbs"
    RELAX_DIR = WORKSPACE / "relaxed_pdbs"
    DOCK_DIR = WORKSPACE / "docking_results"

import shutil

# 1. Create ZIP archives of PDB coordinate folders
print("Archiving PDB structures...")
shutil.make_archive("esmfold_pdbs", "zip", PDB_DIR)
shutil.make_archive("relaxed_pdbs", "zip", RELAX_DIR)
shutil.make_archive("docking_results", "zip", DOCK_DIR)
print("Coordinate ZIP archives created: esmfold_pdbs.zip, relaxed_pdbs.zip, docking_results.zip")

# 2. Sync to Google Drive if mounted
if 'google.colab' in sys.modules:
    drive_out = pathlib.Path("/content/drive/MyDrive/experiment_3/module_3_results")
    drive_out.mkdir(parents=True, exist_ok=True)

    for f in WORKSPACE.glob("*.tsv"):
        shutil.copy(f, drive_out / f.name)
    for z in pathlib.Path(".").glob("*.zip"):
        shutil.copy(z, drive_out / z.name)
    if (WORKSPACE / "module_3_biophysical_summary.png").exists():
        shutil.copy(WORKSPACE / "module_3_biophysical_summary.png", drive_out / "module_3_biophysical_summary.png")

    print(f"All Module 3 artifacts synced to Google Drive: {drive_out}")
else:
    print("Local execution: All artifacts stored in exp3_module_3_workspace/.")
